In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]

In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

In [ ]:
fig, axs, error_stats = plot_empirical_model_error_distribution(
    flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    plot_mode="hist2d",
    split_by_subgroup=True,
    show_split_marginals=True,
    bins=50,
    title="Empirical Error vs Observed Ventilation by Opening Type",
    cmap="magma_r",
    error_xlabel=r"Error; $\sigma_{\Delta C_p}$ Bulk Fit PN",
    ventilation_ylabel="LES Flux Velocity",
    positive_ylabel="Flow Entering",
    negative_ylabel="Flow Exiting",
    print_counts=True,
    colorbar_max=40,
)
display(
    pd.DataFrame(
        [
            {
                "opening_type": opening_type,
                "mu_hat": np.nan if stats is None else stats["mu_hat"],
                "sigma_hat": np.nan if stats is None else stats["sigma_hat"],
                "n": 0 if stats is None else stats["n"],
            }
            for opening_type, stats in error_stats.items()
        ]
    )
)

fig, axs, error_stats = plot_empirical_model_error_distribution(
    flowStatsMI,
    y_var=y_var,
    x_var="q_model-Norm-Adjusted",
    plot_mode="hist2d",
    split_by_subgroup=True,
    show_split_marginals=True,
    bins=50,
    title="Empirical Error vs Observed Ventilation by Opening Type",
    cmap="magma_r",
    error_xlabel=r"Error; $\sigma_{\Delta C_p}$ Bulk Fit PN",
    ventilation_ylabel="LES Flux Velocity",
    positive_ylabel="Flow Entering",
    negative_ylabel="Flow Exiting",
    print_counts=True,
    colorbar_max=40
)


In [ ]:
from pathlib import Path

from emulationHelpers import (
    fit_bayesian_ventilation_q_subgroups,
    fit_bayesian_ventilation_p_subgroups,
    load_bayesian_q_ventilation_fit_results,
    load_bayesian_ventilation_fit_results,
    plot_bayesian_ventilation_q_fit_results,
    plot_bayesian_ventilation_p_fit_results,
    plot_bayesian_ventilation_parameter_posteriors,
    plot_bayesian_ventilation_parameter_qq,
    plot_bayesian_ventilation_parameter_traces,
    save_bayesian_ventilation_fit_results,
    summarize_bayesian_parameter_normal_fit_tests,
    summarize_bayesian_ventilation_fits,
)

a_mu = 1.0
a_sigma = 0.1
p_rms_var = "p_rms-noInt-Norm"
obs_sigma = 0.01

sample_kwargs = {
    "draws": 4000,
    "tune": 4000,
    "chains": 4,
    "cores": 4,
    "progressbar": True,
}

cache_dir = Path("mcmc_cache") / "pressure_scalar_posteriors_log_p_rms"
rerun_mcmc = False

if cache_dir.exists() and not rerun_mcmc:
    bayes_fits = load_bayesian_ventilation_fit_results(cache_dir)
    print(f"Loaded cached Bayesian fits from {cache_dir}")
else:
    bayes_fits = fit_bayesian_ventilation_p_subgroups(
        data=flowStatsMI,
        y_var=y_var,
        x_var=x_var,
        p_rms_var=p_rms_var,
        a_mu=a_mu,
        a_sigma=a_sigma,
        obs_sigma=obs_sigma,
        sample_kwargs=sample_kwargs,
        random_seed=42,
    )
    save_bayesian_ventilation_fit_results(bayes_fits, cache_dir)
    print(f"Saved Bayesian fits to {cache_dir}")


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian Pressure Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
)

bayes_summary = summarize_bayesian_ventilation_fits(bayes_fits)
display(bayes_summary)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.45,
    band_hatch="///",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_posteriors(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
    kde=False,
    overlay_normal=True,
    normal_line_color="tab:blue",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_traces(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
plt.show()


In [ ]:
bayes_normal_fit_shapiro = summarize_bayesian_parameter_normal_fit_tests(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
display(bayes_normal_fit_shapiro)


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_qq(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
)
plt.show()


In [ ]:
# Bayesian q_rms workflow for $\\sigma_{q_n}$ Bulk Measured PN
q_rms_var = "rms-mass_flux-Norm"

q_cache_dir = Path("mcmc_cache") / "q_rms_scalar_posteriors_log_q_rms"
rerun_q_mcmc = False

if q_cache_dir.exists() and not rerun_q_mcmc:
    q_bayes_fits = load_bayesian_q_ventilation_fit_results(q_cache_dir)
    print(f"Loaded cached Bayesian q fits from {q_cache_dir}")
else:
    q_bayes_fits = fit_bayesian_ventilation_q_subgroups(
        data=flowStatsMI,
        y_var=y_var,
        x_var=x_var,
        q_rms_var=q_rms_var,
        a_mu=a_mu,
        a_sigma=a_sigma,
        obs_sigma=obs_sigma,
        sample_kwargs=sample_kwargs,
        random_seed=42,
    )
    save_bayesian_ventilation_fit_results(q_bayes_fits, q_cache_dir)
    print(f"Saved Bayesian q fits to {q_cache_dir}")


In [ ]:
fig, axs = plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian q_rms Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
)

q_bayes_summary = summarize_bayesian_ventilation_fits(q_bayes_fits)
display(q_bayes_summary)
plt.show()


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12), dpi=140, sharex=True, sharey=True)

plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian Pressure Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
    axes=axs[0],
    show_figure_legend=False,
    figure_suptitle=False,
)

plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian q_rms Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
    axes=axs[1],
    show_figure_legend=False,
    figure_suptitle=False,
)

handles, labels = axs[0, 0].get_legend_handles_labels()
legend_items = []
seen = set()
for handle, label in zip(handles, labels):
    if label in (None, "", "_nolegend_") or label in seen:
        continue
    legend_items.append((handle, label))
    seen.add(label)

if legend_items:
    fig.legend(
        [handle for handle, _ in legend_items],
        [label for _, label in legend_items],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.00),
        ncol=5,
        frameon=False,
        fontsize=14,
    )

fig.text(0.02, 0.73, "Pressure", rotation=90, va="center", ha="center", fontsize=18)
fig.text(0.02, 0.28, r"$q_{\mathrm{rms}}$", rotation=90, va="center", ha="center", fontsize=18)
fig.suptitle("Bayesian Pressure and $q_{\mathrm{rms}}$ Scatter Plots", fontsize=22)
plt.tight_layout(rect=[0.04, 0.07, 1, 0.97])
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.45,
    band_hatch="///",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_q_fit_results(
    data=flowStatsMI,
    fit_results=q_bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    compare_normal_mc=True,
    normal_mc_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.35,
    band_hatch="///",
    normal_mc_band_alpha=0.28,
    normal_mc_band_hatch="\\",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_posteriors(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
    kde=False,
    overlay_normal=True,
    normal_line_color="tab:blue",
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_traces(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
plt.show()


In [ ]:
q_bayes_normal_fit_shapiro = summarize_bayesian_parameter_normal_fit_tests(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
display(q_bayes_normal_fit_shapiro)


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_qq(
    q_bayes_fits,
    columns=["a", "q_rms", "sigma_obs"],
)
plt.show()
